# MiniMax H3 生成サーバ (Google Colab)

オープンウェイトの [MiniMax H3](https://huggingface.co/MiniMaxAI/MiniMax-H3) を ComfyUI で走らせ、
キー認証つきの REST API として HTTPS で公開する。手元の `mvc` からはこの API を叩く。

**構成**

```
[mvc / curl] --HTTPS--> [cloudflared] --> [FastAPI :8000 (Bearer認証)] --> [ComfyUI :8188 (127.0.0.1のみ)]
```

ComfyUI 自体には認証が無く、ワークフロー投入は任意コード実行と同義。
ComfyUI は必ず 127.0.0.1 に閉じたままにして、トンネルに出すのは FastAPI だけにする。

**事前準備**

1. ランタイムのタイプを **A100 GPU** にする(ランタイム > ランタイムのタイプを変更)
2. リポジトリの `mvc-h3/` フォルダ一式を Google Drive の直下にアップロードしておく
   (`MyDrive/mvc-h3/` になる)
3. 上から順にセルを実行する

**所要時間とコスト目安**: 初回は ComfyUI 導入 + ウェイト 42.5GB の取得で 15〜25分。
A100 は 5.3 CU/時 (100CU = 1179円) なので **62.5円/時**。実測では 720p・6.6秒の
1本に約16分 = **16.6円**。seedance の同条件 529円に対して約 1/32。
アイドル時間も課金されるので、まとめて生成して落とすこと。

## 1. GPU を確認する

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 2. Drive からコードを取り込む

`mvc-h3/` を丸ごと Drive に置いてある前提。置き場所を変えたら `DRIVE_SRC` を直す。
キーストアやトンネルの認証情報はこの下に置かない。コードを上書きアップロードしたときに
巻き添えで消えないよう、Drive 直下の別ファイルに分けてある。

In [ ]:
import pathlib
import shutil

from google.colab import drive

drive.mount("/content/drive")

DRIVE_SRC = pathlib.Path("/content/drive/MyDrive/mvc-h3")
WORK = pathlib.Path("/content/h3")

if not DRIVE_SRC.exists():
    raise SystemExit(f"{DRIVE_SRC} がありません。mvc-h3/ を Drive にアップロードしてください")

if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(DRIVE_SRC, WORK)
print("配置:", sorted(p.name for p in WORK.iterdir()))

## 3. ComfyUI と依存をインストールする

MiniMax H3 のノードは ComfyUI 0.30.0 以降に同梱されている。

In [ ]:
import pathlib
import subprocess

COMFY = pathlib.Path("/content/ComfyUI")

if not COMFY.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI", str(COMFY)],
        check=True,
    )

!pip install -q -r {COMFY}/requirements.txt
!pip install -q -r /content/h3/server/requirements.txt

version_file = COMFY / "comfyui_version.py"
if version_file.exists():
    print(version_file.read_text().strip())
else:
    print("バージョン情報が読めません。H3 ノードが無ければ ComfyUI を更新すること")

## 4. ウェイトを取得する

| 用途 | ファイル | サイズ |
|---|---|---|
| t2v / i2v | `minimax_h3_fl2va_pruned_int8_convrot` | 21.0 GB |
| 参照生成 (r2v) | `minimax_h3_ref2va_pruned_int8_convrot` | 21.0 GB |
| テキストエンコーダ | `qwen3vl_32b_minimax_h3_nvfp4_awq` | 15.7 GB |
| 映像 VAE | `minimax_h3_video_vae_fp16` | 5.2 GB |
| 音声 VAE | `minimax_h3_audio_vae_fp32` | 0.6 GB |

`fl2va` だけなら 42.5 GB、`ref2va` も入れると 63.5 GB。`/content` は約 78GB なので、
**両方入れると残りは約 5GB**。生成物(1本 2〜4MB)には十分だが、これ以上モデルは足せない。

mvc の i2v は `fl2va` しか使わない。`ref2va` が要るのは、画像や音声をフレームではなく
**参照**として渡したいとき(歌唱リップシンク、キャラ同一性、モーション参照)。
なお両者は 21GB ずつあって A100 40GB に同時には載らないので、交互に呼ぶと毎回入れ替えが
起きる。使うならタスク単位でまとめて投げること。

量子化は **int8 が既定**。fp8 のウェイトもあるが A100 (sm80) は FP8 演算をネイティブ実行できないため、
Ada / Blackwell 以外では int8 のほうが素直に速い。

ランタイムを止めると `/content` は消えるので、次のセッションでは再取得になる。`CACHE` に Drive の
パスを入れておけば再ダウンロードは要らなくなるが、Drive 側に相応の空きが必要で、
読み込み速度は HF から取り直すのと大差ないこともある。まずは空のまま試して、
毎回の待ち時間が気になってから設定するのでよい。

In [ ]:
import subprocess
import sys

TASKS = ["fl2va"]  # 参照生成(歌唱リップシンク等)も使うなら ["fl2va", "ref2va"]
                   # 両方入れると 63.5GB で /content の残りは約 5GB になる
QUANT = "int8"     # int8 | fp8 | bf16
UPSCALER = []      # 基本は空のまま。1080p 出力なら既定の lanczos で足りる
                   # (RealESRGAN は静止画用でフレーム間がちらつく。README「解像度と高精細化」参照)

# Drive にキャッシュするとランタイムを止めても再ダウンロードが要らない。
# ただし相応の空きが必要で、読み込みは HF から取り直すより遅いこともある。
# 空文字にすると毎回 HF から取得する。
CACHE = ""  # 例: "/content/drive/MyDrive/mvc-h3-models"

cmd = [sys.executable, "/content/h3/setup/download_models.py",
       "--comfy", "/content/ComfyUI", "--quant", QUANT, "--tasks", *TASKS]
if UPSCALER:
    cmd += ["--upscaler", *UPSCALER]
if CACHE:
    cmd += ["--cache", CACHE]

subprocess.run(cmd, check=True)
!df -h /content | tail -1

## 5. アクセスキーを発行する

キーストアは Drive 直下の `mvc-h3-keys.json` に置くので、次回のセッションでも同じキーが使える。
平文キーは発行時にしか表示されない。表示された `COLAB_H3_API_KEY=...` を手元の `.env` に貼る。

すでに発行済みなら、このセルは `list` だけ実行すればよい。

In [ ]:
import pathlib

KEYS = pathlib.Path("/content/drive/MyDrive/mvc-h3-keys.json")

!python /content/h3/scripts/genkey.py --keys {KEYS} list

In [ ]:
# 新しいキーが要るときだけ実行する
!python /content/h3/scripts/genkey.py --keys {KEYS} issue --name mvc-local --env

## 6. サーバを起動する

ComfyUI (127.0.0.1:8188) → FastAPI (127.0.0.1:8000) → cloudflared の順に立ち上げる。
ウェイトの読み込みがあるので ComfyUI の初回応答までは数分かかる。

生成した mp4 は `JOBS_DIR` に `20260806-134512_i2v_a1b2c3d4.mp4` の形で貯まる。
既定では Drive に置くので、ランタイムを止めても残る。

In [ ]:
import os
import pathlib
import re
import subprocess
import time

import requests

LOGS = pathlib.Path("/content/logs")
LOGS.mkdir(exist_ok=True)
procs = {}


def spawn(name, args, cwd=None, env=None):
    log = open(LOGS / f"{name}.log", "w")
    proc = subprocess.Popen(args, cwd=cwd, stdout=log, stderr=subprocess.STDOUT, env=env)
    procs[name] = proc
    print(f"{name}: pid={proc.pid}")
    return proc


def wait_for(name, check, timeout, interval=5):
    """check() が真になるまで待つ。プロセスが落ちたらログを見せて止める。"""
    deadline = time.time() + timeout
    while time.time() < deadline:
        if procs[name].poll() is not None:
            print((LOGS / f"{name}.log").read_text()[-3000:])
            raise SystemExit(f"{name} が終了しました")
        try:
            if check():
                return True
        except Exception:
            pass
        time.sleep(interval)
    raise SystemExit(f"{name} が {timeout} 秒以内に応答しませんでした")


# --- ComfyUI (外に出さない) ---
spawn(
    "comfyui",
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
    cwd="/content/ComfyUI",
)
wait_for(
    "comfyui",
    lambda: requests.get("http://127.0.0.1:8188/object_info/MiniMaxH3ImageToVideo", timeout=5).ok,
    timeout=900,
)
print("ComfyUI 準備完了 (MiniMax H3 ノードあり)")

# --- FastAPI ラッパ ---
# 生成した mp4 の保存先。Drive にしておけばランタイムを止めても残る。
JOBS_DIR = "/content/drive/MyDrive/mvc-h3-outputs"

env = dict(os.environ, COMFY_URL="http://127.0.0.1:8188", H3_KEYS_PATH=str(KEYS),
           H3_JOBS_DIR=JOBS_DIR)
spawn(
    "api",
    ["python", "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="/content/h3/server",
    env=env,
)
wait_for("api", lambda: requests.get("http://127.0.0.1:8000/health", timeout=5).ok, timeout=120)
print("API 準備完了")

## 7. HTTPS トンネルを開く

既定は cloudflared の quick tunnel。アカウント不要で `https://....trycloudflare.com` が発行されるが、
URL は起動のたびに変わる。固定したい場合は下のセルの named tunnel 側を使う。

In [ ]:
import re
import time

!wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i -E /content/cloudflared.deb > /dev/null 2>&1

spawn(
    "tunnel",
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
)

PUBLIC_URL = None
deadline = time.time() + 90
while time.time() < deadline and PUBLIC_URL is None:
    text = (LOGS / "tunnel.log").read_text()
    found = re.search(r"https://[\w-]+\.trycloudflare\.com", text)
    if found:
        PUBLIC_URL = found.group(0)
        break
    time.sleep(2)

if PUBLIC_URL is None:
    print((LOGS / "tunnel.log").read_text()[-2000:])
    raise SystemExit("トンネルの URL を取得できませんでした")

print(f"COLAB_H3_ENDPOINT={PUBLIC_URL}")

# URL がログに出てから経路が実際に張られるまで数秒〜十数秒かかる。
# その間 Cloudflare は HTML のエラーページを返すので、JSON が返るまで待つ。
for attempt in range(1, 13):
    try:
        res = requests.get(f"{PUBLIC_URL}/health", timeout=30)
        if res.headers.get("content-type", "").startswith("application/json"):
            print(res.json())
            break
        print(f"[{attempt}/12] {res.status_code} {res.headers.get('content-type')} — 経路待ち")
    except requests.RequestException as exc:
        print(f"[{attempt}/12] {exc}")
    time.sleep(5)
else:
    local = requests.get("http://127.0.0.1:8000/health", timeout=10)
    print("ローカル側:", local.status_code, local.text[:300])
    print((LOGS / "tunnel.log").read_text()[-1500:])
    raise SystemExit("トンネル越しに API へ到達できません")

### (任意) URL を固定する — named tunnel

自分の Cloudflare アカウントとドメインが要る。`cloudflared tunnel login` は対話が必要なので、
手元のマシンで一度トンネルを作り、その認証情報 (`~/.cloudflared/<UUID>.json` と `cert.pem`) を
Drive に置いてから使う。

In [ ]:
# TUNNEL_NAME = "mvc-h3"
# CREDS = "/content/drive/MyDrive/mvc-h3-cloudflared"
# !mkdir -p ~/.cloudflared && cp {CREDS}/* ~/.cloudflared/
# spawn("tunnel", ["cloudflared", "tunnel", "run", "--url", "http://127.0.0.1:8000", TUNNEL_NAME])
# PUBLIC_URL = "https://h3.example.com"  # DNS で割り当てたホスト名

## 8. 疎通テスト

t2v を 1 本だけ生成して、API 全体が通ることを確かめる。
初回はモデルのロードが入るので 5 分前後かかることがある。

In [ ]:
import subprocess
import sys

API_KEY = input("発行したキーを貼ってください: ").strip()

# `!` 行の変数展開は空だと引数ごと消えて分かりにくいエラーになるので、
# ここで明示的に確かめてから subprocess で渡す。
if not globals().get("PUBLIC_URL"):
    raise SystemExit("PUBLIC_URL が空です。トンネルのセルを先に実行してください")
if not API_KEY:
    raise SystemExit("キーが空です")

EXTRA = []
# 1080p で確かめるなら次の行を有効にする(生成解像度が上がる分だけ時間もかかる)
# EXTRA = ["--megapixels", "0.98", "--output-size", "1920x1080"]

subprocess.run(
    [sys.executable, "/content/h3/scripts/smoke_test.py",
     "--endpoint", PUBLIC_URL, "--key", API_KEY, "--out", "/content/smoke.mp4"] + EXTRA,
    check=True,
)

In [ ]:
from IPython.display import Video

Video("/content/smoke.mp4", embed=True, width=640)

## (重要) 使っている間はこの見張りセルを実行したままにする

サーバは `subprocess` で動いているので、起動セルはすぐ終わってしまう。その状態だと
Colab からは「何も実行していないノートブック」に見えて、**アイドル判定
(概ね90分)でランタイムを止められる**。

下のセルは5分ごとに死活を1行出すだけの見張り。実行しっぱなしにしておけば
「実行中」と見なされる。止めたいときはセルを中断すればよい(サーバ自体は止まらない)。

これでも**タブを閉じたりPCがスリープすると切断されうる**。Colab の仕様上、
完全に防ぐ手段は無いので、「止まっても安く復帰できる」ようにしておくのが現実的:

- ウェイトは `CACHE` に Drive のパスを入れて再ダウンロードを避ける
- 生成物は `JOBS_DIR` を Drive にして残す(既定でそうしてある)
- キーは Drive の `mvc-h3-keys.json` なので再発行は不要
- 復帰後は `COLAB_H3_ENDPOINT` の更新だけで済む(quick tunnel は URL が変わる)

無人で長時間回したいなら、Colab ではなく RunPod など API で制御できる環境が向く。

In [ ]:
import time

import requests

# 5分ごとに死活を1行出すだけ。実行したままにしておくと Colab のアイドル判定を受けにくい。
# 中断してもサーバ(subprocess)は止まらない。
while True:
    dead = [name for name, proc in procs.items() if proc.poll() is not None]
    if dead:
        print(f"\n落ちたプロセス: {dead}")
        for name in dead:
            print((LOGS / f"{name}.log").read_text()[-1000:])
        break
    try:
        health = requests.get("http://127.0.0.1:8000/health", timeout=10).json()
        state = f"comfy_ready={health.get('comfy_ready')}"
    except Exception as exc:
        state = f"応答なし: {exc}"
    print(f"{time.strftime('%H:%M:%S')} {state}", flush=True)
    time.sleep(300)

## (任意) サーバのコードだけ入れ替える

`mvc-h3/server/` を直して Drive に上げ直したとき用。**ComfyUI は止めない**ので、
モデルのロード状態(初回 約126秒)がそのまま残る。

手順は「Drive に上げ直す → このセルを実行」だけ。見張りセルを回している場合は
先に中断してから実行し、終わったら見張りを再開する。

In [ ]:
import shutil

# API だけ止める(ComfyUI はそのまま = モデルのロード状態を保つ)
if "api" in procs and procs["api"].poll() is None:
    procs["api"].terminate()
    procs["api"].wait()
    print("api を停止しました")

# Drive から取り込み直す
shutil.rmtree(WORK)
shutil.copytree(DRIVE_SRC, WORK)
print("コードを取り込み直しました:", sorted(p.name for p in WORK.iterdir()))

spawn(
    "api",
    ["python", "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="/content/h3/server",
    env=env,
)
wait_for("api", lambda: requests.get("http://127.0.0.1:8000/health", timeout=5).ok, timeout=120)
print("api を再起動しました")

## 9. ログを見る / 停止する

タブを閉じたりPCがスリープすると、ランタイムが切断されることがある(上の見張りセル参照)。
使い終わったら必ず停止すること(A100 は動かしているだけで課金される)。

In [ ]:
for name in procs:
    print(f"===== {name} =====")
    print((LOGS / f"{name}.log").read_text()[-1500:])

In [ ]:
for name, proc in procs.items():
    if proc.poll() is None:
        proc.terminate()
        print(f"{name} を停止しました")

# ランタイムごと落とす場合は下を実行する
# from google.colab import runtime; runtime.unassign()